# Парсер фильмов с Википедии на Scrapy → CSV (+ рейтинг IMDb)

Стартовые страницы (из задания):
- https://ru.wikipedia.org/wiki/Категория:Фильмы_по_алфавиту  
- https://ru.wikipedia.org/wiki/Категория:Фильмы_по_годам  

Собираю по фильмам:
- **Название**
- **Жанр**
- **Режиссёр**
- **Страна**
- **Год**

Результат сохраняется в **CSV**.

Дополнительно (если хочется): рейтинг IMDb через OMDb (нужен ключ, иначе просто пропускаю).


## **1) Импорты и настройки**
Тут задаю:
- стартовые ссылки
- сколько фильмов собирать
- глубину обхода подкатегорий
- имя CSV файла


In [10]:
import re
import os
import requests

import scrapy
from scrapy.crawler import CrawlerProcess

In [11]:
START_URLS = [
    "https://ru.wikipedia.org/wiki/Категория:Фильмы_по_алфавиту",
    "https://ru.wikipedia.org/wiki/Категория:Фильмы_по_годам",
]

MAX_ITEMS = 300              # ограничение по количеству фильмов
MAX_CATEGORY_DEPTH = 2       # глубина захода в подкатегории
OUT_CSV = "wiki_movies.csv"  # куда сохраняю результат

## **2) Вспомогательные функции**
Небольшие функции для очистки текста, вытаскивания года и IMDb id.


In [12]:
def clean_text(x: str) -> str:
    if x is None:
        return ""
    x = re.sub(r"\s+", " ", x).strip()
    return x

def first_year(text: str) -> str:
    if not text:
        return ""
    m = re.search(r"(18\d{2}|19\d{2}|20\d{2})", text)
    return m.group(1) if m else ""

def extract_imdb_id(urls):
    for u in urls:
        m = re.search(r"imdb\.com/title/(tt\d+)", u)
        if m:
            return m.group(1)
    return ""

## **3) Spider**
Идея простая:
- На страницах категорий беру ссылки на статьи (фильмы) и ссылки на подкатегории.
- У подкатегорий захожу на пару уровней.
- На странице фильма смотрю таблицу справа (infobox) и вытаскиваю нужные поля.

Про IMDb:
- если на странице есть ссылка на IMDb (tt...), то могу достать рейтинг через OMDb API.
- для этого нужен ключ: `OMDB_API_KEY`.
- если ключа нет — `imdb_rating` будет пустой.


In [17]:
class WikiMoviesSpider(scrapy.Spider):
    name = "wiki_movies"
    allowed_domains = ["ru.wikipedia.org", "wikipedia.org"]

    custom_settings = {
        "USER_AGENT": "Mozilla/5.0 (compatible; wiki_movies_student/1.0)",
        "ROBOTSTXT_OBEY": True,
        "DOWNLOAD_DELAY": 0.25,
        "CONCURRENT_REQUESTS": 8,
        "FEEDS": {OUT_CSV: {"format": "csv", "encoding": "utf-8-sig"}},
        "LOG_LEVEL": "INFO",
    }

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.items_count = 0
        self.seen_urls = set()
        self.omdb_key = os.getenv("OMDB_API_KEY", "").strip()

    def start_requests(self):
        for url in START_URLS:
            yield scrapy.Request(url, callback=self.parse_category, meta={"depth": 0})

    def parse_category(self, response):
        depth = response.meta.get("depth", 0)

        # ссылки на статьи в категории
        member_links = response.css("div.mw-category a::attr(href)").getall()
        member_links += response.css("div#mw-pages a::attr(href)").getall()

        for href in member_links:
            if not href or not href.startswith("/wiki/"):
                continue

            # служебные страницы пропускаю
            if ":" in href:
                continue

            # подкатегории отдельно
            if href.startswith("/wiki/Категория:"):
                continue

            url = response.urljoin(href)
            if url in self.seen_urls:
                continue
            self.seen_urls.add(url)

            yield scrapy.Request(url, callback=self.parse_movie)

        # подкатегории (заходим ограниченно по глубине)
        if depth < MAX_CATEGORY_DEPTH:
            subcats = response.css("div#mw-subcategories a::attr(href)").getall()
            for href in subcats:
                if not href or not href.startswith("/wiki/Категория:"):
                    continue
                url = response.urljoin(href)
                if url in self.seen_urls:
                    continue
                self.seen_urls.add(url)
                yield scrapy.Request(url, callback=self.parse_category, meta={"depth": depth + 1})

        # "Следующая страница" в категории
        next_page = response.xpath("//a[contains(., 'Следующая страница')]/@href").get()
        if next_page:
            yield response.follow(next_page, callback=self.parse_category, meta={"depth": depth})

    def _get_infobox_value(self, response, keys):
        for key in keys:
            xpath = f"//table[contains(@class,'infobox')]//tr[th[contains(normalize-space(.), '{key}')]]/td"
            text = " ".join(response.xpath(xpath + "//text()").getall())
            text = clean_text(text)
            if text:
                return text
        return ""

    def parse_movie(self, response):
        if self.items_count >= MAX_ITEMS:
            return

        title = clean_text(
            response.css("h1#firstHeading .mw-page-title-main::text").get()
            or " ".join(response.css("h1#firstHeading *::text").getall())
            or response.css("title::text").get()
        )
        title = title.replace(" — Википедия", "").strip()

        genre = self._get_infobox_value(response, ["Жанр", "Жанры"])
        director = self._get_infobox_value(response, ["Режиссёр", "Режиссер"])
        country = self._get_infobox_value(response, ["Страна", "Страны"])
        year_raw = self._get_infobox_value(response, ["Год", "Год выхода", "Дата выхода"])
        year = first_year(year_raw) or first_year(title)

        # IMDb id
        links = response.css("a.external::attr(href)").getall()
        imdb_id = extract_imdb_id(links)

        imdb_rating = ""
        if imdb_id and self.omdb_key:
            try:
                r = requests.get(
                    "https://www.omdbapi.com/",
                    params={"i": imdb_id, "apikey": self.omdb_key},
                    timeout=10
                )
                data = r.json()
                imdb_rating = str(data.get("imdbRating", "")).strip()
                if imdb_rating == "N/A":
                    imdb_rating = ""
            except Exception:
                imdb_rating = ""

        self.items_count += 1

        yield {
            "title": title,
            "genre": genre,
            "director": director,
            "country": country,
            "year": year,
            "imdb_id": imdb_id,
            "imdb_rating": imdb_rating,
            "wiki_url": response.url,
        }

## **IMDb рейтинг через OMDb**
Теперь `imdb_id` берётся **через Wikidata** (свойство IMDb ID = P345), это надёжнее, чем искать внешнюю ссылку на IMDb на странице статьи.


In [18]:
import os

os.environ["OMDB_API_KEY"] = "14d7f845"


## **4) Запуск (чтобы в Jupyter не было ReactorNotRestartable)**

В Jupyter бывает ошибка `ReactorNotRestartable`, если запускать Scrapy через `CrawlerProcess().start()` и потом запускать ещё раз в том же ядре.

Чтобы не ловить эту проблему, я запускаю паука **в отдельном процессе**:
1) записываю паука в файл `wiki_movies_spider.py`
2) запускаю `python wiki_movies_spider.py` — он сам сохранит CSV


In [15]:
# Запуск в отдельном процессе (можно запускать сколько угодно раз)
# Если нужно больше фильмов:
# import os; os.environ["MAX_ITEMS"] = "1000"

!python wiki_movies_spider.py


Готово. CSV: wiki_movies.csv


2026-01-30 18:09:47 [scrapy.utils.log] INFO: Scrapy 2.12.0 started (bot: scrapybot)
2026-01-30 18:09:47 [scrapy.utils.log] INFO: Versions: lxml 5.2.2.0, libxml2 2.11.7, cssselect 1.2.0, parsel 1.9.1, w3lib 2.2.1, Twisted 24.10.0, Python 3.10.2 (tags/v3.10.2:a58ebcc, Jan 17 2022, 14:12:15) [MSC v.1929 64 bit (AMD64)], pyOpenSSL 24.2.1 (OpenSSL 3.3.2 3 Sep 2024), cryptography 43.0.1, Platform Windows-10-10.0.19045-SP0
2026-01-30 18:09:47 [scrapy.addons] INFO: Enabled addons:
[]
2026-01-30 18:09:47 [scrapy.extensions.telnet] INFO: Telnet Password: fed8d43d7b74e736
2026-01-30 18:09:47 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.feedexport.FeedExporter',
 'scrapy.extensions.logstats.LogStats']
2026-01-30 18:09:47 [scrapy.crawler] INFO: Overridden settings:
{'CONCURRENT_REQUESTS': 8,
 'DOWNLOAD_DELAY': 0.25,
 'LOG_LEVEL': 'INFO',
 'ROBOTSTXT_OBEY': True,
 'USER_AGENT': 'Mozilla/5.0 (com

## **5) Проверка результата**
Просто читаю CSV и смотрю первые строки.


In [16]:
import pandas as pd

# keep_default_na=False чтобы пустые строки не превращались в NaN (так проще понять, что реально записалось)
res = pd.read_csv(OUT_CSV, keep_default_na=False)
print("Размер результата:", res.shape)

# быстрая проверка: сколько пустых title
empty_titles = (res["title"].astype(str).str.strip() == "").sum()
print("Пустых title:", int(empty_titles))

display(res.head(10))

Размер результата: (223, 8)
Пустых title: 0


,title,genre,director,country,year,imdb_id,imdb_rating,wiki_url
0,«Чудотворец» из Бирюлёва,игровое кино,Борис Эпштейн,СССР,1958,,,https://ru.wikipedia.org/wiki/%C2%AB%D0%A7%D1%...
1,«SOS» над тайгой,"драма , приключения","Аркадий Кольцатый , Валентин Перов",СССР,1976,tt0191415,5.4,https://ru.wikipedia.org/wiki/%C2%ABSOS%C2%BB_...
2,«Спартак». Действующие лица и… болельщики,"документальный , спорт","Илья Гутман , Иосиф Пастернак",СССР,1985,,,https://ru.wikipedia.org/wiki/%C2%AB%D0%A1%D0%...
3,"«Москвич», любовь моя",драма,Арам Шахбазян,Армения Франция Россия,2015,tt3835106,7.6,https://ru.wikipedia.org/wiki/%C2%AB%D0%9C%D0%...
4,…которого любили все,документально - биографический фильм,Леонид Осыка,СССР,1982,tt21977052,,https://ru.wikipedia.org/wiki/%E2%80%A6%D0%BA%...
5,…И прекрасный миг победы,спортивный фильм,Вячеслав Винник,СССР,1984,tt0262477,,https://ru.wikipedia.org/wiki/%E2%80%A6%D0%98_...
6,…и передайте привет ласточкам,"драма, военный",Яромил Йиреш,Чехословакия,1972,tt0122844,6.7,https://ru.wikipedia.org/wiki/%E2%80%A6%D0%B8_...
7,38-я параллель (фильм),драма боевик военный,Кан Джегю,Республика Корея,2004,tt0386064,8.0,https://ru.wikipedia.org/wiki/38-%D1%8F_%D0%BF...
8,"Мария, мать Иисуса",библейский фильм драма,.mw-parser-output .ts-Wikidata-redLink a{backg...,США,1999,tt0214930,5.4,https://ru.wikipedia.org/wiki/%D0%9C%D0%B0%D1%...
9,36 безумных кулаков,боевик комедия,Чэнь Чжихуа,Гонконг,1977,tt0076661,5.0,https://ru.wikipedia.org/wiki/36_%D0%B1%D0%B5%...



    return g.throw(self.value.with_traceback(self.tb))
  File "C:\Users\gerasimov\AppData\Local\Programs\Python\Python310\lib\site-packages\scrapy\core\downloader\middleware.py", line 68, in process_request
    return (yield download_func(request, spider))
twisted.internet.error.ConnectError: An error occurred while connecting: 10065: Сделана попытка выполнить операцию на сокете для недоступного хоста..
2026-01-30 22:44:30 [scrapy.downloadermiddlewares.retry] ERROR: Gave up retrying <GET https://ru.wikipedia.org/wiki/6_%D0%B4%D0%BD%D0%B5%D0%B9> (failed 3 times): An error occurred while connecting: 10065: Сделана попытка выполнить операцию на сокете для недоступного хоста..
2026-01-30 22:44:30 [scrapy.core.scraper] ERROR: Error downloading <GET https://ru.wikipedia.org/wiki/6_%D0%B4%D0%BD%D0%B5%D0%B9>
Traceback (most recent call last):
  File "C:\Users\gerasimov\AppData\Local\Programs\Python\Python310\lib\site-packages\twisted\internet\defer.py", line 2013, in _inlineCallbacks
    resu